# HC-DPO: Colab runner

Before running: set the runtime to an **L4 or A100 GPU** (Runtime > Change runtime type), and add a Colab secret named `WANDB_API_KEY` with notebook access turned on.

The project folder lives on Google Drive, so data, checkpoints and results survive disconnects. After a disconnect, rerun the setup cell and then the same run cell: finished runs are skipped and an interrupted run resumes.

In [ ]:
# 1. Setup: mount Drive first, then install. Edit PROJECT_DIR if your folder is elsewhere.
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Projects/hc-dpo'
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = f'{PROJECT_DIR}/src'
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Pre-flight (a few minutes): tests, then a tiny train + evaluate with the debug config.
!python -m pytest tests -q -p no:cacheprovider
!python -m hcdpo.data --config configs/debug.yaml
!python -m hcdpo.train --config configs/debug.yaml
!python -m hcdpo.evaluate --config configs/debug.yaml --base
!python -m hcdpo.evaluate --config configs/debug.yaml

In [ ]:
# 3. Smoke test of the real config: 20 steps at full batch size, to check memory and speed on this GPU.
!python -m hcdpo.data --config configs/dpo.yaml
!python -m hcdpo.train --config configs/dpo.yaml --set max_steps=20 run_name=smoke wandb.enabled=false

In [ ]:
# 4. DPO baseline sweep: base-model evaluation, then beta in {0.05, 0.1, 0.25, 0.5}.
# If the smoke test ran out of memory, use: !bash scripts/run_dpo_sweep.sh batch_size=4 grad_accum_steps=8
!bash scripts/run_dpo_sweep.sh

In [ ]:
# 5. HC-DPO sweep. Choose four epsilons spanning the kl_mean values of the DPO runs.
!EPSILONS="1 2 4 8" bash scripts/run_hcdpo_sweep.sh

In [ ]:
# 6. Seeds 42, 43, 44 for the best DPO beta and HC-DPO at that run's final KL (its kl_mean).
!bash scripts/run_seeds.sh 0.1 2.0